# IRF Convolution and Realistic TCSPC Simulation

This notebook constructs a realistic time-correlated single-photon counting
(TCSPC) simulation by combining the instrument response function (IRF),
a monoexponential fluorescence model, numerical convolution, detector
background, and Poisson photon-counting noise.

The workflow is assembled from reusable toolkit components rather than from
one specialized simulation function:

1. generate and normalize a Gaussian IRF;
2. optionally shift the IRF in time;
3. generate an ideal fluorescence decay;
4. convolve the fluorescence decay with the IRF;
5. add detector background after convolution;
6. sample measured photon counts from a Poisson distribution.

The notebook also investigates how the simulated signal depends on:

- IRF width;
- fluorescence lifetime;
- temporal alignment;
- observation-window boundaries;
- time-bin resolution.

## Learning objectives

By the end of this notebook, we will be able to:

- construct a normalized Gaussian instrument response function;
- interpret the effect of IRF normalization;
- apply positive and negative temporal shifts to an IRF;
- generate an ideal monoexponential fluorescence decay;
- convolve the fluorescence signal with the IRF;
- keep fluorescence background and detector background conceptually separate;
- generate Poisson-distributed TCSPC counts;
- identify when IRF broadening significantly distorts a lifetime measurement;
- recognize boundary, truncation, alignment, and time-bin-resolution effects.

## 1. Scientific model

An ideal monoexponential fluorescence decay is described by

$$
I_{\mathrm{fl}}(t)
=
A \exp\left(-\frac{t}{\tau}\right),
$$

where:

- $A$ is the fluorescence amplitude;
- $\tau$ is the fluorescence lifetime.

A real TCSPC instrument does not record this ideal signal directly. Its
finite temporal response broadens the fluorescence decay. The broadened signal
is represented by the convolution

$$
I_{\mathrm{conv}}(t)
=
[\mathrm{IRF} * I_{\mathrm{fl}}](t).
$$

A constant detector background is then added:

$$
\lambda(t)
=
I_{\mathrm{conv}}(t) + B,
$$

where $B$ is the expected background count level per time bin.

Finally, the measured counts are sampled according to Poisson statistics:

$$
N_i \sim \operatorname{Poisson}(\lambda_i).
$$

This notebook combines the individual toolkit components into a realistic TCSPC simulation workflow:

$$
\text{decay model}
\rightarrow
\text{IRF convolution}
\rightarrow
\text{background}
\rightarrow
\text{Poisson sampling}.
$$

The order of these operations matters. The detector background is added after
convolution because it is not fluorescence emission and therefore should not
be broadened by the instrument response. The workflow remains explicit rather
than being hidden inside a separate IRF-specific simulation function.

## 2. Imports

We use NumPy for numerical arrays, Matplotlib for visualization, and the
public TCSPC Toolkit API for decay generation, IRF processing, convolution,
and Poisson sampling.

The notebook does not reproduce the implementation of these functions. Its
purpose is to demonstrate how the existing components can be composed into a
realistic simulation workflow. The notebook uses the following TCSPC Toolkit APIs:

- `monoexponential_decay()` to generate the ideal fluorescence model;
- `generate_gaussian_irf()` to construct a Gaussian IRF;
- `normalize_irf()` to give the IRF unit area;
- `shift_irf()` to introduce a temporal offset;
- `convolve_decay_with_irf()` to model instrument broadening;
- `sample_photon_counts()` to generate Poisson-distributed measurements.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from tcspc_toolkit.convolution import convolve_decay_with_irf
from tcspc_toolkit.irf import (
    generate_gaussian_irf,
    normalize_irf,
    shift_irf,
)
from tcspc_toolkit.models import monoexponential_decay
from tcspc_toolkit.simulation import sample_photon_counts

## 3. Simulation settings

We first define one common set of parameters for the main simulation.

The time interval must be long enough to include both the IRF and most of the
fluorescence decay. A fine time grid is useful because numerical convolution
and sub-bin IRF shifts depend on the temporal sampling resolution.

The fixed random seed value makes the Poisson realization reproducible.

All times in this notebook are expressed in nanoseconds.

The initial example uses:

- a 10 ns observation window;
- 1001 time points;
- a fluorescence lifetime of 1 ns;
- a Gaussian IRF with a full width at half maximum of 0.3 ns;
- a detector background of 5 expected counts per bin.

The IRF centre is placed away from the left boundary so that most of its area remains inside the observation window.

In [2]:
start_time = 0.0
end_time = 10.0
n_bins = 1001

amplitude = 1_000.0
lifetime = 1.0
background = 5.0

irf_centre = 1.0
irf_fwhm = 0.3
irf_amplitude = 1.0
irf_shift = 0.0

random_seed = 42

In [4]:
print(f"Time interval: {start_time:.2f}–{end_time:.2f} ns")
print(f"Number of bins: {n_bins}")
print(f"Fluorescence amplitude: {amplitude:.1f}")
print(f"Fluorescence lifetime: {lifetime:.2f} ns")
print(f"Detector background: {background:.2f} counts/bin")
print(f"IRF centre: {irf_centre:.2f} ns")
print(f"IRF FWHM: {irf_fwhm:.2f} ns")
print(f"Random seed: {random_seed}")

Time interval: 0.00–10.00 ns
Number of bins: 1001
Fluorescence amplitude: 1000.0
Fluorescence lifetime: 1.00 ns
Detector background: 5.00 counts/bin
IRF centre: 1.00 ns
IRF FWHM: 0.30 ns
Random seed: 42
